# Moaddab, Wright & McDannald (2020) on CRCNS pag-2

Paper: Moaddab M, Wright KM, McDannald MA. Early adolescent adversity alters periaqueductal
gray/dorsal raphe threat responding in adult female rats. Sci Rep 10, 18035 (2020),
doi:10.1038/s41598-020-74457-3.

Dataset: CRCNS pag-2, doi:10.6080/K0833Q7F, read from the public bucket at
`https://crcnsarchive.s3-us-west-2.amazonaws.com/pag-2/live/`. Version: `unversioned`. CRCNS data
sets carry no version numbers, so the only published state is the current one under `live/` (no
`old/` folder exists). It is pinned by the MD5 sums of the data set's `checksums.md5`:
data.zip `7d1686e3d4f6f36895c38f34eda632d4`, docs/crcns_pag-2_data_description.pdf
`9f6c40904e9123fe8ff488d6006c8df4`; the first cell checks that the object read has that MD5.
The data description is version 0.6 (April 4, 2021).

Claims: `claims/crcns-pag-2/10.1038-s41598-020-74457-3.yaml`, written from the version of record
(PMC7582948) without its supplement. Methods details the claims file lacks are taken from the paper
and, where the paper leaves them open, from the authors' own script and cube in the data set
(`data/vlpagAnalysis.m`, `data/cubeFile/cube.mat`), as each section says.

| Claim | Tested |
|---|---|
| fig1bc-adversity-slows-weight-gain | blocked (no body weights in the data set) |
| fig1ef-acquisition-equivalent-final | blocked (no pre-surgery sessions in the data set) |
| fig2b-baseline-poke-equivalent | tested |
| fig2c-recording-discrimination | tested |
| fig2d-cue-responsive-counts | tested |
| fig2d-higher-proportion-responsive | tested |
| fig2ef-baseline-firing | tested |
| fig4ab-excited-temporal-pattern | tested without bregma (no bregma level per rat) |
| fig4c-excited-inflated-onset | tested |
| fig4de-inhibited-temporal-pattern | tested without bregma |
| fig4f-inhibited-danger-decrease | tested |
| fig4f-inhibited-danger-uncertainty-discrimination | tested |
| fig5abc-excited-reduced-threat-probability | tested |
| fig5d-threat-tuning-curve | tested |
| figs4-peak-assignment-same-pattern | tested |
| fig5efg-inhibited-fear-output | tested |
| fig6-excited-firing-threat-decoupled | tested |
| figs5-excited-firing-fear-output-unrelated | tested |
| fig7-inhibited-firing-fear-output | tested |
| figs6-inhibited-firing-threat-probability | tested |

In [ ]:
import time
import collections
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import pag2

t0 = time.time()
etag, md5_ok = pag2.check_md5()
print("data.zip ETag", etag, "matches checksums.md5:", md5_ok)
zf = pag2.open_zip()
members = [i for i in zf.infolist() if not i.is_dir()]
print(len(members), "files in data.zip;", sum(i.filename.startswith("data/matFiles/") for i in members), "unit files")
for i in members:
    if not i.filename.startswith("data/matFiles/"):
        print("  ", i.filename, i.file_size)
cube, names = pag2.load_cube(zf)
U = pag2.load_all(zf, names)
print(len(U), "units loaded in %.0f s" % (time.time() - t0))
grp = np.array([u["group"] for u in U])

## Data check

What the files hold: each unit file has the spike times (`spike`), the event times `EVT01` to
`EVT14` and the mean waveform. The cube, `vlPAGcube`, has for every unit a trial table (`tag`: cue
type 1 to 4 and onset time per trial), binned raw, differential and Z scored firing, nose poke
rates and suppression ratios (`poke`, `cer`), and the waveform. The cell below lists every field
of the cube with its shape, type and range, identifies the event codes against the cube's trial
table, and checks the cube's stored values against values computed here from the spikes and
pokes.

In [ ]:
def walk(o, path=""):
    if hasattr(o, "_fieldnames"):
        for f in o._fieldnames:
            walk(getattr(o, f), path + "." + f)
    elif isinstance(o, np.ndarray) and o.dtype == object and o.size == 1:
        walk(o.flat[0], path)
    else:
        a = np.asarray(o)
        rng = (a.min(), a.max()) if a.dtype.kind in "fiu" and a.size else ""
        print(f"vlPAGcube{path:32s} {str(a.shape):16s} {str(a.dtype):8s} {rng}")
walk(cube)

# units, rats, sessions
rats = sorted(set(u["rat"] for u in U))
print("\nrats:", {g: [r for r in rats if pag2.group_of(r) == g] for g in ("Con", "EAA")})
print("rats in the script's lists but not in the data:",
      sorted(set(pag2.CON_RATS + pag2.EAA_RATS) - set(rats)))
print("units per group:", dict(collections.Counter(grp)))
sessions = {}
for u in U:
    sessions.setdefault(u["session"], u)
print("sessions (rat and date):", len(sessions), dict(collections.Counter(s["group"] for s in sessions.values())),
      "; the paper and the data description state 72 Control and 107 EAA")
ev_sets = {tuple(np.round(u["onset"], 3)) for u in U}
print("distinct trial tables (cue onset times):", len(ev_sets))
progs = collections.defaultdict(set)
for u in U:
    progs[u["session"]].add(u["name"][10:16])
tab_keys = collections.defaultdict(set)
for u in U:
    tab_keys[tuple(np.round(u["onset"], 2))].add(u["session"])
print("sessions with more than one behaviour program:", sum(len(v) > 1 for v in progs.values()),
      "; trial tables shared by two sessions:", sum(len(v) > 1 for v in tab_keys.values()))
print("units per rat:", dict(collections.Counter(u["rat"] for u in U)))
print("behaviour programs:", dict(collections.Counter(u["name"][10:16] for u in U)))
hashes = collections.Counter((u["n_spikes"], u["counts"].sum()) for u in U)
dups = [k for k, v in hashes.items() if v > 1]
print("pairs of units with the same spike count and binned counts (possible duplicates):", len(dups))

# event codes against the cube's trial table
tag = pag2.cube_field(cube, "tag")
ok_onset = ok_type = 0
for j, u in enumerate(U):
    tt = np.array([{"danger": 1, "uncertainty": 2, "safety": 4}[x] for x in u["type"]])
    tt[(u["type"] == "uncertainty") & ~u["shock"]] = 3
    ok_onset += np.allclose(u["onset"], tag[:, 1, j])
    ok_type += np.array_equal(tt, tag[:, 0, j])
print(f"\ncue onsets equal to the cube's tag in {ok_onset}/{len(U)} units, trial types in {ok_type}/{len(U)}")
code_map = collections.Counter()
for u in U:
    for code in ("EVT09", "EVT11", "EVT12"):
        t = u["events"][code][0]
        code_map[(u["name"][10:16], code, u["type"][np.argmin(np.abs(u["onset"] - t))])] += 1
for k, v in sorted(code_map.items()):
    print("  ", k, v, "units")
print("trials per unit:", dict(collections.Counter(len(u["onset"]) for u in U)),
      "; per type:", dict(collections.Counter(u["type"]).items()) if False else
      dict(collections.Counter(tuple(sorted(collections.Counter(u["type"]).items())) for u in U)))
print("shocks per unit:", dict(collections.Counter(int(u["shock"].sum()) for u in U)),
      "; shocked uncertainty trials per unit:",
      dict(collections.Counter(int(u["shock"][u["type"] == "uncertainty"].sum()) for u in U)))
lag14 = np.concatenate([np.min(np.abs(u["events"]["EVT14"][None, :] - (u["onset"][:, None] + 31.5)), axis=1) for u in U])
print("EVT14 within 0.01 s of 31.5 s after a cue onset: %.1f%% of trials (median deviation %.3f s, max %.1f s)"
      % (100 * np.mean(lag14 < 0.01), np.median(lag14), lag14.max()))
# how far around each cue onset the spikes and the nose pokes are stored
for what, items in (("spikes", [(u["spikes"], u["onset"]) for u in U]),
                    ("nose pokes (EVT13)", [(u["events"]["EVT13"], u["onset"]) for u in sessions.values()])):
    aft, bef = map(np.concatenate, zip(*(pag2.time_from_onsets(t, on) for t, on in items)))
    w30 = (aft < 30) | (bef <= 20)
    w33 = (aft < 33.5) | (bef <= 20.5)
    tail = aft[(bef > 20.5) & np.isfinite(aft)]
    print(f"{what}: {aft.size}; within 20 s before to 30 s after a cue onset {100 * w30.mean():.2f}%, within 20.5 s before"
          f" to 33.5 s after {100 * w33.mean():.3f}% ({int((~w33).sum())} outside); time after the latest onset, outside the 20.5 s"
          f" before the next: 99th percentile {np.percentile(tail, 99):.1f} s, 99.9th {np.percentile(tail, 99.9):.1f} s;"
          f" largest time before the first cue onset of a session {bef[~np.isfinite(aft)].max():.1f} s")
gaps = np.concatenate([np.diff(np.sort(u["onset"])) for u in sessions.values()])
print("gap between consecutive cue onsets: %.1f to %.1f s, mean %.1f s" % (gaps.min(), gaps.max(), gaps.mean()))

# the authors' script: what it holds that the analysis could use
script = zf.read("data/vlpagAnalysis.m").decode("latin-1").lower()
print("\nvlpagAnalysis.m:", len(script.splitlines()), "lines; occurrences of",
      {w: script.count(w) for w in ("bregma", "outlier", "exclu", "regress(", "corrcoef", "corr(")})
script_lines = zf.read("data/vlpagAnalysis.m").decode("latin-1").splitlines()
def show(a, b):
    for k in range(a, b + 1):
        print(f"{k:5d}  {script_lines[k - 1]}")
print("\nthe interval regressions of Control cue-excited neurons (the other groups and types follow the same pattern):")
show(2149, 2150); show(2162, 2171)
print("\nthe per-neuron mean firing of cue-inhibited neurons:")
show(2796, 2817)
import re
print("\nper-neuron window means of cue-inhibited firing anywhere in the script:",
      sorted(set(re.findall(r"inh\.fire\.z\.ms250\.(\w*(?:firstfive|lastfive|cueentire)mean)", script))))

In [ ]:
# The cube's stored values against values computed from the spikes and pokes
B10 = pag2.cube_field(cube, "fire.raw.base10")[:, 0, :].astype(float)
CSEL = {w: pag2.cube_field(cube, "fire.raw.select." + k)[:, 0, :].astype(float)
        for w, k in (("first1", "cueOnset"), ("first5", "cueFirstFive"), ("last5", "cueLastFive"))}
exact = np.array([pag2.window_rate(u, "base10") for u in U]).T
d = exact - B10
print("fire.raw.base10 minus exact 10 s rate: %.2f to %.2f Hz; cube equals floor(exact) in %.1f%% of trials"
      % (-d.max(), -d.min(), 100 * np.mean(np.floor(exact + 1e-9) == B10)))
print("cube rates stored as", pag2.cube_field(cube, "fire.raw.base10").dtype,
      "; units with a mean 10 s baseline under 1 Hz:", int(np.sum(exact.mean(0) < 1)),
      "; trials whose stored baseline is 0 Hz while spikes were fired:", int(np.sum((B10 == 0) & (exact > 0))))
cz = pag2.cube_field(cube, "fire.z.s1.interval")
cd = pag2.cube_field(cube, "fire.diff.s1.interval").astype(float)
err_d = err_z = 0.0
for j, u in enumerate(U):
    r1 = u["counts"].reshape(32, 50, 4).sum(2).astype(float)          # spikes per 1 s bin = Hz
    own = r1[:, 18:20].mean(1, keepdims=True)
    err_d = max(err_d, np.abs(r1 - own - cd[:, :, j]).max())
    z = (cd[:, :, j] - cd[:, :, j].mean()) / cd[:, :, j].std(ddof=1)
    err_z = max(err_z, np.abs(z - cz[:, :, j]).max())
print("fire.diff.s1 = 1 s rate minus the trial's own 2 s baseline, within %.2f Hz (stored as integers)" % err_d)
print("fire.z.s1 = fire.diff.s1 Z scored over all trials and bins of the unit (ddof 1), within %.1g" % err_z)

# nose pokes and suppression ratios
pb = pag2.cube_field(cube, "poke.base")[:, 0, :].astype(float)
pc = pag2.cube_field(cube, "poke.cue")[:, 0, :].astype(float)
csr = pag2.cube_field(cube, "cer.cue")[:, 0, :]
mine_pb = np.array([u["poke_base"] / 20 * 60 for u in U]).T
mine_pc = np.array([u["poke_cue"] / 10 * 60 for u in U]).T
print("\npoke.base equals pokes per minute in the 20 s before cue onset in %.1f%% of trials" % (100 * np.mean(np.isclose(pb, mine_pb))))
for w in (2, 10):
    alt = np.array([(np.searchsorted(u["events"]["EVT13"], u["onset"]) - np.searchsorted(u["events"]["EVT13"], u["onset"] - w)) / w * 60 for u in U]).T
    print("   with a %d s window: %.1f%%" % (w, 100 * np.mean(np.isclose(pb, alt))))
print("poke.cue equals pokes per minute in the 10 s cue in %.1f%% of trials" % (100 * np.mean(np.isclose(pc, mine_pc))))
sr = np.array([pag2.suppression_ratio(u, "drop") for u in U]).T
empty = np.isnan(sr)
print("cer.cue equals (base - cue)/(base + cue) with the 20 s baseline in %.2f%% of trials with pokes;"
      % (100 * np.mean(np.isclose(sr[~empty], csr[~empty]))),
      "trials with no poke in baseline and cue:", int(empty.sum()), "(unit-trials), stored as", np.unique(csr[empty]))
nsess_empty = collections.Counter()
for s, u in sessions.items():
    nsess_empty[u["group"]] += int(np.isnan(pag2.suppression_ratio(u, "drop")).sum())
print("empty trials over the 182 sessions:", dict(nsess_empty), "of", {g: 32 * n for g, n in collections.Counter(s["group"] for s in sessions.values()).items()})
print("suppression ratio range %.2f to %.2f; firing in 250 ms bins 0 to %.0f Hz; spike count per unit %d to %d"
      % (np.nanmin(sr), np.nanmax(sr), max(pag2.rates(u).max() for u in U), min(u["n_spikes"] for u in U), max(u["n_spikes"] for u in U)))

Clock check. The spikes and pokes are stored only around the trials (printed below and above), so
the onsets cannot be moved into the inter-trial interval. Cue responses should appear at the cue
onsets of the trial table and vanish when the onsets are moved 10 s earlier, so that every window
falls before the cue; the selection rule (reading b, exact rates) is run at both, and the rates in
the shifted baseline are printed to show that the spikes cover it. For the pokes, the danger minus
safety suppression ratio is compared between the true labels and labels permuted within each
session (seeded generator).

In [ ]:
e, i = pag2.SELECTION_READINGS["b"]
true_lab = np.array([pag2.select(u, e, i) for u in U])
shift_lab = np.array([pag2.select(u, e, i, shifted=True) for u in U])
print("cue-responsive units at the true onsets:", int(np.sum(true_lab != "osc")), "of", len(U),
      "; at onsets 10 s earlier:", int(np.sum(shift_lab != "osc")),
      "(chance under the rule: about 6 tests x 2 tails x 0.0083/2 = %.1f%% of units)" % (100 * (1 - (1 - 0.0083) ** 6)))
print("mean rate in the 10 s baseline: true %.2f Hz, shifted (20 to 10 s before onset) %.2f Hz"
      % (np.mean([pag2.window_rate(u, "base10").mean() for u in U]), np.mean([pag2.window_rate(u, "base10", True).mean() for u in U])))
rng_clock = np.random.default_rng(1)
dsr = []
for s, u in sessions.items():
    a = pag2.suppression_ratio(u, "zero")
    perm = rng_clock.permutation(u["type"])
    dsr.append((a[u["type"] == "danger"].mean() - a[u["type"] == "safety"].mean(),
                a[perm == "danger"].mean() - a[perm == "safety"].mean()))
dsr = np.array(dsr)
print("danger minus safety suppression ratio per session, mean: true labels %.3f, permuted labels %.3f; sessions with danger > safety: %d and %d of %d"
      % (dsr[:, 0].mean(), dsr[:, 1].mean(), np.sum(dsr[:, 0] > 0), np.sum(dsr[:, 1] > 0), len(dsr)))

Findings of the data check, stated with the numbers above:

- The 354 units (151 Control, 203 EAA) and 20 rats (8 Control, 12 EAA) match the paper. No two
  units share their spike counts, so no recording is deposited twice. Cue responses are present at
  the trial table's onsets and nearly absent 10 s earlier, and fear follows the cue labels, which is
  consistent with the spikes, pokes and trial table sharing a clock. The
  sessions do not: the units come from 182 sessions (rat and date, each with its own trial table),
  74 Control and 108 EAA, where the paper and the data description state 72 and 107. No session
  holds more than one behaviour program and no two sessions share a trial table, so the extra
  sessions are not duplicates; the difference is not explained.
- The group of each rat is given only by the lists in the authors' script, not by the data
  description or the files.
- The event codes are not documented. EVT09, EVT11 and EVT12 are the cue onsets, but which code is
  danger, uncertainty or safety changes with the behaviour program (epr38a, b, c); EVT08 holds the
  shocks, EVT13 the nose pokes, EVT14 a marker about 31.5 s after each cue onset. Spikes and nose pokes
  are stored only around the trials, from about 20 s before to about 33.5 s after each cue onset,
  not over the whole session: 99.98% of spikes and 99.99% of pokes fall within 20.5 s before to
  33.5 s after an onset, and 4.5% of spikes and 5.4% of pokes fall outside 20 s before to 30 s
  after.
- The cube stores raw firing rates as integers truncated towards zero (uint8), so a unit firing
  below 1 Hz has stored rates of 0.
- `poke.base` is the poke rate per minute in the 20 s before cue onset, not "2 s prior to cue
  presentation" as the data description says, and it is per minute, a unit the description does not
  give.
- `cer.cue` stores 1 for the trials with no poke in baseline or cue (a ratio of 0/0).
- The data description gives the uncertainty shock rows a shock probability of 0.25; the paper and
  the description's own summary give 0.375.
- The data set holds no body weights, no pre-surgery discrimination sessions, and no bregma level
  per rat (the description gives only how many rats were at each of four levels). The authors'
  script does not mention bregma and has no step that excludes neurons (no occurrence of
  "bregma", "outlier" or "exclu").

## Blocked claims

`fig1bc-adversity-slows-weight-gain` needs the body weights from P24 to P55, and
`fig1ef-acquisition-equivalent-final` the eight pre-surgery discrimination sessions. Neither is in
the data set: its files are the 354 recording-session units, the cube built from them, the script
and three cue sounds (listed in the first cell), and the data description lists nothing else.
Both are blocked as `data_missing`.

## fig2b-baseline-poke-equivalent

> Control and EAA rats had equivalent baseline nose poke rates during recording sessions.

One baseline rate per rat, from the 20 s before cue onset (Methods, "Calculating suppression
ratios"), over its recording sessions; each session is counted once although several units share
it. Readings: the mean of the rat's session means, and the mean over all its trials pooled.
Two-sided independent samples t test.

In [ ]:
def per_rat(fn):
    out = {}
    for s, u in sessions.items():
        out.setdefault(u["rat"], []).append(fn(u))
    return out

rats_sorted = sorted(set(u["rat"] for u in U))
rgrp = np.array([pag2.group_of(r) for r in rats_sorted])
base = per_rat(lambda u: u["poke_base"] / 20 * 60)
readings_bp = {
    "mean of session means": np.array([np.mean([x.mean() for x in base[r]]) for r in rats_sorted]),
    "all trials pooled": np.array([np.concatenate(base[r]).mean() for r in rats_sorted]),
}
for k, v in readings_bp.items():
    t = stats.ttest_ind(v[rgrp == "Con"], v[rgrp == "EAA"])
    print(f"{k:22s} Con {v[rgrp=='Con'].mean():.2f} (sd {v[rgrp=='Con'].std(ddof=1):.2f}) EAA {v[rgrp=='EAA'].mean():.2f} "
          f"(sd {v[rgrp=='EAA'].std(ddof=1):.2f}) pokes/min; t({t.df:.0f}) = {t.statistic:.2f}, p = {t.pvalue:.3f}")
print("per rat (pokes/min):", {r: round(v, 1) for r, v in zip(rats_sorted, readings_bp["mean of session means"])})
print("sessions per rat:", {r: len(base[r]) for r in rats_sorted})

**Result: reproduced.** Under both readings the groups do not differ (t(18) = 1.19, p = 0.25; the
paper gives t = 1.24, p = 0.23); every session has 32 trials, so the two readings give the same
values.

## fig2c-recording-discrimination

> During recording sessions both groups showed high fear to danger, less to uncertainty and least
> to safety, with a main effect of cue, no group by cue interaction, and each cue pair differing in
> each group.

Per rat, the mean suppression ratio for each cue over its recording sessions (each session once).
Mixed ANOVA, cue within, group between; paired t tests within each group. Readings: rat value as
the mean of session means or over trials pooled; trials with no pokes (0/0) set to 0, dropped, or
set to 1 as the cube stores them.

In [ ]:
def sr_rat(empty, comb):
    vals = per_rat(lambda u: (pag2.suppression_ratio(u, empty), u["type"]))
    Y = []
    for r in rats_sorted:
        if comb == "session means":
            Y.append([np.mean([np.nanmean(s[t == ty]) for s, t in vals[r]]) for ty in pag2.TYPES])
        else:
            s = np.concatenate([x[0] for x in vals[r]]); t = np.concatenate([x[1] for x in vals[r]])
            Y.append([np.nanmean(s[t == ty]) for ty in pag2.TYPES])
    return np.array(Y)

for empty in ("zero", "drop", "one"):
    for comb in ("session means", "pooled"):
        Y = sr_rat(empty, comb)
        a = pag2.mixed_anova(Y, rgrp)
        line = f"empty={empty:4s} {comb:13s} cue F({a['A'][1]},{a['A'][2]}) = {a['A'][0]:.1f}, p = {a['A'][3]:.1e}; " \
               f"group x cue F = {a['group:A'][0]:.2f}, p = {a['group:A'][3]:.2f};"
        for g in ("Con", "EAA"):
            x = Y[rgrp == g]
            t1 = stats.ttest_rel(x[:, 0], x[:, 1]); t2 = stats.ttest_rel(x[:, 1], x[:, 2])
            line += f" {g} D-U t = {t1.statistic:.2f} p = {t1.pvalue:.4f}, U-S t = {t2.statistic:.2f} p = {t2.pvalue:.1e};"
        print(line)
        if empty == "zero" and comb == "session means":
            print("   means (danger, uncertainty, safety):", {g: Y[rgrp == g].mean(0).round(3) for g in ("Con", "EAA")},
                  "range per rat %.2f to %.2f" % (Y.min(), Y.max()))

**Result: reproduced.** Under every reading the cue effect is large (F(2,36) from 179.5 to 194.5,
p <= 1.9e-19), the group by cue interaction is absent (p from 0.84 to 0.90), and all four paired tests
have p <= 0.0091 with danger > uncertainty > safety in both groups.

## fig2d-cue-responsive-counts

> 32 of 151 Control neurons (21.2%) and 49 of 203 EAA neurons (24.1%) were cue-excited; 48 of 151
> Control neurons (31.8%) and 84 of 203 EAA neurons (41.4%) were cue-inhibited.

Paired t tests (two-sided, p < 0.0083) of the raw rate in the 10 s before cue onset against each
cue epoch, per cue, under the three readings of the claims file. Two sources of the raw rates are
run: the exact rates from the spike times, and the rates stored in the authors' cube
(`fire.raw.base10` and `fire.raw.select.cueOnset`, `cueFirstFive`, `cueLastFive`), which are
truncated to whole Hz. The authors' script runs its selection on the cube's stored rates
(vlpagAnalysis.m, lines 9 to 67, MATLAB `ttest(x, y)`, a paired test, at alpha 0.00833) with
reading (b) (lines 136 to 147 and 225 to 236).

In [ ]:
cube_rates = [dict(base10=B10[:, j], **{w: CSEL[w][:, j] for w in CSEL}) for j in range(len(U))]
labels = {}
for rd, (e, i) in pag2.SELECTION_READINGS.items():
    labels[rd, "exact"] = np.array([pag2.select(u, e, i) for u in U])
    labels[rd, "cube"] = np.array([pag2.select(u, e, i, r=cube_rates[j]) for j, u in enumerate(U)])
expected = {("Con", "exc"): 32, ("EAA", "exc"): 49, ("Con", "inh"): 48, ("EAA", "inh"): 84}
print("reading  rates   Con exc  EAA exc  Con inh  EAA inh  (paper 32, 49, 48, 84; tolerance 10%)")
for (rd, src), L in labels.items():
    cnt = {k: int(np.sum((grp == k[0]) & (L == k[1]))) for k in expected}
    within = all(abs(cnt[k] - v) <= 0.1 * v for k, v in expected.items())
    print(f"   {rd}     {src:6s}  " + "  ".join(f"{cnt[k]:7d}" for k in expected) + f"   all within tolerance: {within}")
print("unit totals:", dict(collections.Counter(grp)))
lab = labels["b", "cube"]          # the selection used by the claims that follow
lab_exact = labels["b", "exact"]
print("\nselection used below: reading (b), cube rates. Agreement with exact rates, reading (b):",
      int(np.sum(lab == lab_exact)), "of", len(U), "units;", collections.Counter(zip(lab, lab_exact)))
for g in ("Con", "EAA"):
    print(g, "by rat:", {r: dict(collections.Counter(lab[[u['rat'] == r for u in U]])) for r in rats_sorted if pag2.group_of(r) == g})

**Result: reproduced.** With the cube's stored rates and reading (b), the counts are 32, 49, 48
and 84, the paper's exactly, and the totals are 151 and 203. With exact rates from the spike times,
reading (c) is also within tolerance (30, 48, 51, 89), while reading (b) gives 30, 50, 54 and 87,
the Control inhibited count outside the tolerance. Reading (a) is outside under both rate sources
(Control inhibited 44 and 34). The paper's counts follow exactly from the cube's truncated rates
and the script's reading (b), which the claims that follow use.

## fig2d-higher-proportion-responsive

> A higher proportion of EAA neurons than Control neurons were cue-responsive (133/203, 65.5%,
> against 80/151, 53.0%).

Pearson chi-square without continuity correction on the 2 by 2 table, under every reading of the
selection.

In [ ]:
for (rd, src), L in labels.items():
    tab = np.array([[np.sum((grp == g) & (L != "osc")), np.sum((grp == g) & (L == "osc"))] for g in ("Con", "EAA")])
    chi, p, *_ = stats.chi2_contingency(tab, correction=False)
    chiy, py, *_ = stats.chi2_contingency(tab, correction=True)
    fr = tab[:, 0] / tab.sum(1)
    print(f"reading {rd} {src:5s}: Con {tab[0,0]}/{tab[0].sum()} = {fr[0]:.3f}, EAA {tab[1,0]}/{tab[1].sum()} = {fr[1]:.3f}; "
          f"chi2 = {chi:.2f}, p = {p:.3f} (Yates {chiy:.2f}, p = {py:.3f})")

**Result: reproduced.** Under the selection used here (b, cube rates) the fractions are 0.530 and
0.655 and chi-square = 5.68, p = 0.017, as in the paper; with exact rates (b) they are 0.556 and
0.675, chi-square = 5.19, p = 0.023, also within tolerance.

## fig2ef-baseline-firing

> Baseline firing rates did not differ between Control and EAA neurons of any functional type,
> while across groups baseline firing differed by functional type and was lowest for cue-excited
> neurons.

Per neuron, the mean raw rate over all trials in (a) the 10 s or (b) the 2 s before cue onset, from
the spike times. Independent samples t tests per type, two-sided; one-way ANOVA over the three
types, groups pooled.

In [ ]:
for src, L in (("cube (b)", lab), ("exact (b)", lab_exact)):
    for w in ("base10", "base2"):
        bf = np.array([pag2.window_rate(u, w).mean() for u in U])
        tt = []
        for ty in ("exc", "inh", "osc"):
            r = stats.ttest_ind(bf[(L == ty) & (grp == "Con")], bf[(L == ty) & (grp == "EAA")])
            tt.append(f"{ty} t({r.df:.0f}) = {r.statistic:.2f} p = {r.pvalue:.2f}")
        f = stats.f_oneway(*[bf[L == ty] for ty in ("exc", "inh", "osc")])
        means = {ty: round(bf[L == ty].mean(), 2) for ty in ("exc", "inh", "osc")}
        print(f"selection {src}, {w}: " + "; ".join(tt) + f"; type F(2,{len(U)-3}) = {f.statistic:.2f}, p = {f.pvalue:.4f}; means (Hz) {means}")
print("baseline rate range: %.2f to %.2f Hz" % (bf.min(), bf.max()))

**Result: reproduced.** With the selection used here, no group difference has p < 0.05 (smallest p
= 0.26, all |t| < 1.13) under either baseline window (with the exact-rate selection, smallest p =
0.12), the type effect has F(2,351) = 6.44, p =
0.0018 (10 s) and 6.78, p = 0.0013 (2 s), and cue-excited neurons have the lowest mean (13.9 Hz,
against 22.0 and 25.3). The paper's error degrees of freedom are 346, five fewer than the 351 the
354 neurons give.

## Normalized firing

As the opening comment of the claims file and the Methods ("Z score normalization") give it: per
neuron and cue (danger, uncertainty with its 16 trials, safety), the trial-mean rate in 250 ms
bins minus that cue's mean 2 s pre-cue rate, Z scored over all 600 bins of the neuron. The
authors' script instead averages the cube's trial-wise Z scores (fire.z.ms250, each trial's own
baseline subtracted and Z scored over all trials and bins); this is run as a variant for the
population ANOVAs. The figure shows the population means with the selection used here.

In [ ]:
NF = np.array([pag2.normalized_firing(u)[0] for u in U])            # neurons x 3 cues x 200 bins
Z250 = pag2.cube_field(cube, "fire.z.ms250.interval")
NFc = np.zeros_like(NF)
for j, u in enumerate(U):
    for k, ty in enumerate(pag2.TYPES):
        NFc[j, k] = Z250[u["type"] == ty, :, j].mean(0)
tbin = (np.arange(200) - 80) * 0.25
fig, ax = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for a_, (pop, g) in zip(ax.flat, [(p, g) for p in ("exc", "inh") for g in ("Con", "EAA")]):
    m = (lab == pop) & (grp == g)
    for k, (ty, col) in enumerate(zip(pag2.TYPES, ("red", "purple", "blue"))):
        a_.plot(tbin[72:128], NF[m, k, 72:128].mean(0), color=col, label=ty)
    a_.axvline(0, color="k", lw=0.5); a_.axvline(10, color="k", lw=0.5)
    a_.set_title(f"{g} {pop} (n = {m.sum()})"); a_.set_ylabel("Z")
ax[0, 0].legend(); ax[1, 0].set_xlabel("s from cue onset"); ax[1, 1].set_xlabel("s from cue onset")
plt.tight_layout(); plt.show()
print("plotted means range %.2f to %.2f; all points inside the axes" % (min(NF[(lab == p)][:, :, 72:128].mean(0).min() for p in ("exc", "inh")),
      max(NF[(lab == p) & (grp == g)][:, :, 72:128].mean(0).max() for p in ("exc", "inh") for g in ("Con", "EAA"))))

## fig4ab-excited-temporal-pattern

> Control and EAA cue-excited neurons differed in the temporal pattern of their cue-selective
> firing, a group by cue by bin interaction that held with bregma recording level as a factor.

Mixed ANOVA of the normalized firing in the 56 bins from 2 s before cue onset to 2 s after offset,
cue and bin within, group between. The model with bregma level cannot be run: the data set gives
no bregma level per rat, and the authors' script does not mention bregma (data check).

In [ ]:
def pop_anova(pop, L=None, F=None):
    L = lab if L is None else L
    F = NF if F is None else F
    m = L == pop
    a = pag2.mixed_anova(F[m][:, :, pag2.WIN["anova"]], grp[m])
    return a, m

for name, L, F in (("main (cube-rate selection, Methods normalization)", lab, NF),
                   ("variant: the script's normalization", lab, NFc),
                   ("variant: exact-rate selection", lab_exact, NF)):
    a, m = pop_anova("exc", L, F)
    f = a["group:AB"]
    print(f"{name}: n = {dict(collections.Counter(grp[m]))}; group x cue x bin F({f[1]},{f[2]}) = {f[0]:.2f}, p = {f[3]:.2g};"
          f" cue x bin F = {a['AB'][0]:.2f}, p = {a['AB'][3]:.2g}")

**Result: partially reproduced.** The group by cue by bin interaction is significant, F(110,8690)
= 1.30, p = 0.019 (the paper: F(110,8360) = 3.32, p = 3e-28), weaker than the paper's; the
script's normalization gives F = 1.40, p = 0.004, and the exact-rate selection F = 1.29, p =
0.021. The part with bregma level could not be tested, as neither the data set nor the authors' script
gives a bregma level per rat, so the claim holds only in the part that could be tested.

## fig4c-excited-inflated-onset

> In the first 1 s of the cue, EAA cue-excited neurons fired more than Control neurons (main
> effect of group), with a significant group difference to safety; both groups still discriminated
> danger from uncertainty and uncertainty from safety.

Per neuron, the mean normalized firing over the four bins of the first 1 s of each cue. Mixed
ANOVA (cue within, group between); independent samples t tests per cue at 0.05/3; paired t tests
within each group at p < 0.025.

In [ ]:
def window_tests(pop, win, L=None, F=None, show=True):
    L = lab if L is None else L
    F = NF if F is None else F
    m = L == pop
    V = F[m][:, :, pag2.WIN[win]].mean(2); g = grp[m]
    a = pag2.mixed_anova(V, g)
    out = dict(group=a["group"], gxc=a["group:A"], means={gg: V[g == gg].mean(0) for gg in ("Con", "EAA")})
    out["t"] = {ty: stats.ttest_ind(V[g == "Con", k], V[g == "EAA", k]) for k, ty in enumerate(pag2.TYPES)}
    out["paired"] = {gg: {pair: stats.ttest_rel(V[g == gg, pair[0]], V[g == gg, pair[1]]) for pair in ((0, 1), (1, 2))} for gg in ("Con", "EAA")}
    if show:
        print(f"group F({a['group'][1]},{a['group'][2]}) = {a['group'][0]:.2f}, p = {a['group'][3]:.4f}; group x cue F = {a['group:A'][0]:.2f}, p = {a['group:A'][3]:.3f}")
        print("means (danger, uncertainty, safety):", {k: v.round(3) for k, v in out["means"].items()},
              "; range per neuron %.2f to %.2f" % (V.min(), V.max()))
        for ty, r in out["t"].items():
            print(f"   {ty:11s} Con vs EAA t({r.df:.0f}) = {r.statistic:.2f}, p = {r.pvalue:.4f}")
        for gg, d in out["paired"].items():
            print(f"   {gg} paired: " + "; ".join(f"{pag2.TYPES[p[0]][:3]}-{pag2.TYPES[p[1]][:3]} t({r.df:.0f}) = {r.statistic:.2f}, p = {r.pvalue:.4f}" for p, r in d.items()))
        for gg in ("Con", "EAA"):
            byrat = collections.defaultdict(list)
            for j in np.where(m & (grp == gg))[0]:
                byrat[U[j]["rat"]].append(F[j, :, pag2.WIN[win]].mean(1))
            print(f"   {gg} by rat (n, mean danger/uncertainty/safety):", {r: (len(v), np.round(np.mean(v, 0), 2).tolist()) for r, v in byrat.items()})
    return out

print("main"); o = window_tests("exc", "first1")
print("\nvariant, the script's normalization"); window_tests("exc", "first1", F=NFc)
print("\nvariant, exact-rate selection"); window_tests("exc", "first1", L=lab_exact)

**Result: reproduced.** EAA cue-excited neurons fire more in the first 1 s (group F(1,79) = 5.19,
p = 0.025); the safety difference is t(79) = -3.46, p = 0.0009, below 0.0167, with EAA higher; the
paired tests give danger > uncertainty (Control p = 0.0047, EAA p = 0.023) and uncertainty >
safety (p <= 0.0018) in both groups. Both variants meet every part. With the script's
normalization the statistics are those of the paper (group F(1,79) = 4.96 against the paper's
4.96; safety t = -3.60 against 3.61).

## fig4de-inhibited-temporal-pattern

> Control and EAA cue-inhibited neurons differed in the temporal pattern of their cue firing (a
> group by cue by bin interaction that held with bregma level as a factor). Control neurons
> decreased firing equally to danger and uncertainty with little change to safety, sustained over
> the cue; EAA neurons decreased most to danger, less to uncertainty, mostly in the first half of
> the cue.

In [ ]:
for name, L, F in (("main", lab, NF), ("variant: the script's normalization", lab, NFc), ("variant: exact-rate selection", lab_exact, NF)):
    a, m = pop_anova("inh", L, F)
    f = a["group:AB"]
    print(f"{name}: n = {dict(collections.Counter(grp[m]))}; group x cue x bin F({f[1]},{f[2]}) = {f[0]:.2f}, p = {f[3]:.2g}")
m = lab == "inh"
for g in ("Con", "EAA"):
    x = NF[m & (grp == g)]
    print(g, "mean normalized firing (danger, uncertainty, safety): first half",
          x[:, :, pag2.WIN["first5"]].mean((0, 2)).round(2), "second half", x[:, :, pag2.WIN["last5"]].mean((0, 2)).round(2))

**Result: partially reproduced.** The interaction is significant, F(110,14300) = 1.54, p =
0.00024 (the paper: F(110,14190) = 1.69, p = 9e-6), and under both variants (p = 4.4e-6 and
2.5e-5). The population means follow the described pattern: Control danger and uncertainty
decrease alike over both halves of the cue (-0.75 and -0.61 in the first half, -0.69 and -0.67 in
the second) with safety near zero, while EAA decreases most to danger in the first half (-1.06,
uncertainty -0.79) and the two are alike in the second (-0.74, -0.76). The part with bregma level
could not be tested (no bregma level per rat in the data set or the script), so the claim holds only in the part that could be tested.

## fig4f-inhibited-danger-decrease

> In the first 5 s of the cue, EAA cue-inhibited neurons decreased firing more than Control neurons
> (main effect of group), with a greater decrease to danger; the group difference to uncertainty
> fell short of the corrected level.

In [ ]:
print("main"); o5 = window_tests("inh", "first5")
print("\nvariant, the script's normalization"); window_tests("inh", "first5", F=NFc)
print("\nvariant, exact-rate selection"); window_tests("inh", "first5", L=lab_exact)

**Result: reproduced.** Group F(1,130) = 9.15, p = 0.003, with EAA lower; danger t(130) = 2.58, p =
0.011, below 0.0167, with EAA lower; uncertainty t(130) = 1.76, p = 0.080, short of the corrected
level as in the paper. Both variants meet the criterion; the script's normalization gives the
paper's statistics (group F = 10.81 against 10.82, danger t = 2.69 against 2.70, uncertainty t =
2.39, p = 0.0185 against 2.38, p = 0.019).

## fig4f-inhibited-danger-uncertainty-discrimination

> In the first half of the cue, EAA cue-inhibited neurons discriminated danger from uncertainty,
> while Control cue-inhibited neurons did not.

Paired t tests of the first 5 s firing to danger and uncertainty within each group, judged at
0.05/2 and 0.05/3.

In [ ]:
for gg in ("EAA", "Con"):
    r = o5["paired"][gg][(0, 1)]
    print(f"{gg}: danger {o5['means'][gg][0]:.3f}, uncertainty {o5['means'][gg][1]:.3f}; t({r.df:.0f}) = {r.statistic:.2f}, p = {r.pvalue:.4f};"
          f" below 0.025: {r.pvalue < 0.025}, below 0.0167: {r.pvalue < 0.05/3}")
for name, kw in (("script's normalization", dict(F=NFc)), ("exact-rate selection", dict(L=lab_exact))):
    ov = window_tests("inh", "first5", show=False, **kw)
    print(name + ":", {gg: (round(ov["paired"][gg][(0, 1)].statistic, 2), round(ov["paired"][gg][(0, 1)].pvalue, 4)) for gg in ("EAA", "Con")})

**Result: reproduced.** EAA neurons fire less to danger than to uncertainty (t(83) = -3.21, p =
0.0019, below both corrected levels; the paper: t = 3.14, p = 0.002); Control neurons do not differ
(t(47) = -1.29, p = 0.20; the paper: t = 1.40, p = 0.17). The script's normalization gives the
paper's values (EAA t = -3.14, p = 0.0024; Control t = -1.40, p = 0.17).

## Single-unit regression

Per neuron, the 32 trials of its session; the response is the trial's normalized firing in each
1 s interval of the cue; the regressors are the trial's suppression ratio for the 10 s cue (fear
output) and the cue's shock probability, 1, 0.375 or 0 (threat probability), with a constant
(Methods, "Single-unit, linear regression"). The claims file names readings of the trial-wise
normalized firing and of the 0/0 trials:

- firing (a): trial rate minus its cue's mean 2 s baseline, Z scored with the neuron's
  normalization; (b): the same with the trial's own baseline; (c): the trial's rate Z scored over
  all trials and bins of the neuron;
- 0/0 trials: set to 0, or dropped.

Two more readings come from the authors' script, which regresses the cube's trial-wise Z score
(`vlPAGcube.fire.z.s1.interval`, columns 21 to 30) on the cube's suppression ratio
(`vlPAGcube.cer.cue`) (vlpagAnalysis.m, lines 2162 to 2171, printed in the data check): firing (d), the cube's Z score, and
0/0 trials set to 1, as the cube stores them. All twelve combinations are run. The two window
readings of the claims file (mean of the interval betas, or one regression on the window's mean
firing) give the same beta (see `pag2.interval_betas`).

The rule, used for every claim that rests on the regressions (Figs. 5, S4, 6, S5, 7, S6): the
outcome is judged under one combination, the script's own (d, 0/0 trials set to 1), the same for
every claim and not chosen claim by claim. The paper leaves the trial-wise firing open
and the authors' code in the data set settles it, so this follows the code (author, step 1.6), as
an assumption. The section below shows that this combination also gives the paper's printed
statistics more closely than any other. Each claim's section also gives the outcome under the
claims file's own rule (reproduced under one of its six readings, a to c with 0/0 trials set to 0
or dropped); where the two differ, the outcome is marked as resting on the assumption.

For the population firing (Figs. 4, 6, 7), the Methods fix the normalization (the trial-averaged
differential firing is Z scored), the claims file follows them, and the script's different
normalization is a variant (reader, step 2: where the Methods and the code disagree, the Methods
decide and the code's choice is a variant). It is run as a full variant, under every regression
reading.

In [ ]:
ZS1 = pag2.cube_field(cube, "fire.z.s1.interval")
READINGS = [(f, e) for f in "abcd" for e in ("zero", "drop", "one")]
ADOPTED = ("d", "one")
resp = lab != "osc"

def all_betas(unc=0.375, L=None):
    L = lab if L is None else L
    out = {}
    for rd in READINGS:
        FO = np.full((len(U), 10), np.nan); TP = np.full((len(U), 10), np.nan)
        for j, u in enumerate(U):
            if L[j] == "osc":
                continue
            FO[j], TP[j] = pag2.interval_betas(u, rd[0], rd[1], unc, ZS1[:, :, j])
        out[rd] = (FO, TP)
    return out

BETAS = all_betas()
nan_fits = {rd: int(np.isnan(v[0][resp]).any(1).sum()) for rd, v in BETAS.items()}
print("neurons regressed:", int(resp.sum()), "; fits with a missing beta under any reading:", sum(nan_fits.values()))

### Neurons in the regressions

The paper's error degrees of freedom imply fewer neurons than the selection gives (Fig. 5A to C,
t(66) and F(1,65), about 68 and 67 of 81 cue-excited neurons; Fig. 5D, F(1,78), 80). The cell below
counts the neurons that a rule might drop: those whose session has trials without pokes (0/0),
fits with a missing beta, and designs that are not of full rank. The script's searches above find
no exclusion step in it.

In [ ]:
nempty = np.array([int(np.isnan(pag2.suppression_ratio(u, "drop")).sum()) for u in U])
for pop in ("exc", "inh"):
    for g in ("Con", "EAA"):
        mm = (lab == pop) & (grp == g)
        print(f"{pop} {g}: {mm.sum()} neurons; {int(np.sum(mm & (nempty > 0)))} from sessions with a 0/0 trial "
              f"({int(nempty[mm].sum())} unit-trials); {mm.sum() - int(np.sum(mm & (nempty > 0)))} without")
def full_rank(u, empty):
    X = np.column_stack([np.ones(32), pag2.suppression_ratio(u, empty), [0.375 if t == "uncertainty" else pag2.PROB[t] for t in u["type"]]])
    X = X[~np.isnan(X).any(1)]
    return np.linalg.matrix_rank(X) == 3
for pop in ("exc", "inh"):
    idx_ = np.where(lab == pop)[0]
    print(pop, "betas missing under any reading:", sum(int(np.isnan(BETAS[rd][0][idx_]).any(1).sum()) for rd in READINGS),
          "; designs not of full rank (0/0 set to 1, dropped):", sum(not full_rank(U[j], "one") for j in idx_), sum(not full_rank(U[j], "drop") for j in idx_))

None of these gives the paper's counts. 24 of the 81 cue-excited neurons (17 Control, 7 EAA) come
from sessions with a trial without pokes, no beta is missing under any reading, and every
cue-excited design is of full rank (one cue-inhibited design is not; see the section on it before the summary). Dropping the neurons with a
0/0 trial would leave 57 cue-excited neurons, not 67 or 68. Which neurons the paper left out stays
open, and the tests use all selected neurons.

### Which regression reading the paper's statistics follow

For each of the twelve combinations, the statistics the paper prints for Figs. 5A to C, 5D and 5E
to G, and the Control R2 of Fig. 6 (with the script's normalization of the firing, which gives the
paper's population statistics), against the paper's values. Absolute values of t are compared, as
the paper prints them unsigned. The last column is the mean relative deviation from the paper's
values.

In [ ]:
GRID = np.arange(9) * 0.125
TUNE = {}
for rd in READINGS:
    TUNE[rd] = np.array([[pag2.interval_betas(U[j], rd[0], rd[1], v, ZS1[:, :, j])[1].mean() for v in GRID] for j in np.where(lab == "exc")[0]])
PAPER = {"5ABC GxR F": 6.80, "5ABC TP t": 2.71, "5ABC FO t": 0.17, "5D assign F": 18.88, "5D group F": 4.01, "5D GxA F": 1.25,
         "5EFG GxRxI F": 2.14, "5EFG FO t": 2.31, "5EFG TP t": 0.81, "6 Con dan R2": 0.56, "6 Con unc R2": 0.22}
def paper_stats(rd):
    FO, TP = BETAS[rd]
    out = {}
    me = lab == "exc"; ge = grp[me]
    a = pag2.mixed_anova(np.stack([FO[me], TP[me]], 1), ge)
    tp, fo = TP[me].mean(1), FO[me].mean(1)
    out["5ABC GxR F"] = a["group:A"][0]
    out["5ABC TP t"] = abs(stats.ttest_ind(tp[ge == "Con"], tp[ge == "EAA"]).statistic)
    out["5ABC FO t"] = abs(stats.ttest_ind(fo[ge == "Con"], fo[ge == "EAA"]).statistic)
    a = pag2.mixed_anova(TUNE[rd], ge)
    out["5D assign F"], out["5D group F"], out["5D GxA F"] = a["A"][0], a["group"][0], a["group:A"][0]
    mi = lab == "inh"; gi = grp[mi]
    a = pag2.mixed_anova(np.stack([FO[mi], TP[mi]], 1), gi)
    out["5EFG GxRxI F"] = a["group:AB"][0]
    fo, tp = FO[mi][:, 5:].mean(1), TP[mi][:, 5:].mean(1)
    out["5EFG FO t"] = abs(stats.ttest_ind(fo[gi == "Con"], fo[gi == "EAA"]).statistic)
    out["5EFG TP t"] = abs(stats.ttest_ind(tp[gi == "Con"], tp[gi == "EAA"]).statistic)
    tpw = TP[:, :10].mean(1)
    for k, ty in ((0, "dan"), (1, "unc")):
        mm = me & (grp == "Con")
        out[f"6 Con {ty} R2"] = stats.pearsonr(NFc[mm, k, pag2.WIN["first1"]].mean(1), tpw[mm])[0] ** 2
    return out
print("reading   " + " ".join(f"{k:>12s}" for k in PAPER) + "  mean rel. dev.")
print("paper     " + " ".join(f"{v:12.2f}" for v in PAPER.values()))
DEV = {}
for rd in READINGS:
    s = paper_stats(rd)
    DEV[rd] = np.mean([abs(s[k] - v) / v for k, v in PAPER.items()])
    print(f"{rd[0]} {rd[1]:4s}    " + " ".join(f"{s[k]:12.2f}" for k in PAPER) + f"  {DEV[rd]:.3f}")
print("closest to the paper:", min(DEV, key=DEV.get), "; adopted:", ADOPTED)

The script's combination (d, 0/0 trials set to 1) gives the paper's statistics most closely: mean
relative deviation 0.076, against 0.167 for the next, (d, 0/0 dropped), and 0.236 to 0.724 for the
claims file's six readings. Under it the Fig. 5D statistics are F = 18.69, 4.15 and 1.34 (the paper
18.88, 4.01 and 1.25), the Fig. 5E to G statistics F = 2.15, t = 2.34 and 0.82 (the paper 2.14, 2.31
and 0.81) and the Fig. 6 Control R2 0.57 and 0.22 (the paper 0.56 and 0.22). Its Fig. 5A to C
statistics are further from the paper's (F = 3.79 and t = 2.12 against 6.80 and 2.71), where the
paper's degrees of freedom show that about 13 neurons were left out. This supports the adopted
combination as the one the paper computed; the combination itself is taken from the script (lines
2162 to 2171), not from this comparison.

## fig5abc-excited-reduced-threat-probability

> Control cue-excited neurons signaled threat probability (positive betas from cue onset, rising
> over the cue) with fear output betas near zero; EAA cue-excited neurons showed lower threat
> probability betas and similar fear output betas, a group by regressor interaction.

Mixed ANOVA of the betas (regressor and interval within, group between); t tests of the whole-cue
mean betas between groups.

In [ ]:
def fig5abc(B, L=None):
    L = lab if L is None else L
    rows = {}
    m = L == "exc"; g = grp[m]
    for rd, (FO, TP) in B.items():
        a = pag2.mixed_anova(np.stack([FO[m], TP[m]], 1), g)
        tp, fo = TP[m].mean(1), FO[m].mean(1)
        t1 = stats.ttest_ind(tp[g == "Con"], tp[g == "EAA"]); t2 = stats.ttest_ind(fo[g == "Con"], fo[g == "EAA"])
        full = a["group:A"][3] < 0.05 and t1.pvalue < 0.05 and t1.statistic > 0 and t2.pvalue >= 0.05
        part = (not full) and t1.pvalue < 0.05 and t1.statistic > 0
        rows[rd] = "reproduced" if full else "partial" if part else "not"
        print(f"{rd[0]} {rd[1]:4s}: group x regressor F({a['group:A'][1]},{a['group:A'][2]}) = {a['group:A'][0]:.2f}, p = {a['group:A'][3]:.3f}; "
              f"TP Con {tp[g=='Con'].mean():.3f} EAA {tp[g=='EAA'].mean():.3f} t = {t1.statistic:.2f} p = {t1.pvalue:.3f}; "
              f"FO Con {fo[g=='Con'].mean():.3f} EAA {fo[g=='EAA'].mean():.3f} t = {t2.statistic:.2f} p = {t2.pvalue:.3f} -> {rows[rd]}")
    return rows

r5 = fig5abc(BETAS)
FO, TP = BETAS[ADOPTED]
m = lab == "exc"
for g in ("Con", "EAA"):
    print(g, "adopted reading, mean betas per 1 s interval: TP", TP[m & (grp == g)].mean(0).round(2), "FO", FO[m & (grp == g)].mean(0).round(2))

In [ ]:
print("variant, exact-rate selection")
r5x = fig5abc(all_betas(L=lab_exact), L=lab_exact)

**Result: partially reproduced.** Under the adopted reading (d, 0/0 trials set to 1) the whole-cue
threat probability beta is higher in Control (0.953 against 0.558, t = 2.12, p = 0.037) and the
fear output betas do not differ (p = 0.87), but the group by regressor interaction falls short
(F(1,79) = 3.79, p = 0.055). Of the six combinations the claims file names, none meets all three
parts; (b) with 0/0 trials dropped gives the same partial pattern. The script's firing with 0/0
trials set to 0 meets all three (interaction p = 0.031), but that combination is not the adopted one
and is further from the paper's statistics (section above). The claims file's own rule (one of its
six readings) gives the same outcome, partial, so the outcome does not rest on the assumption. Control threat probability betas are
positive from the first interval and rise over the cue (0.72 to 1.22). With the exact-rate
selection no combination meets the threat probability part (smallest p = 0.060). The paper's
degrees of freedom (F(1,65), t(66)) imply about 67 of the 81 neurons; which were left out is not
said.

## fig5d-threat-tuning-curve

> The threat tuning curve of cue-excited neurons peaked at an uncertainty assignment of 0.500 in
> both groups; the EAA curve had the same shape but was lower across all assignments (main effect
> of assignment and of group, no interaction).

Nine regressions per neuron with the uncertainty value 0 to 1 in steps of 0.125, the threat
probability beta averaged over the 10 intervals (Methods, "Threat probability tuning curve").

In [ ]:
m = lab == "exc"; g = grp[m]
for rd in READINGS:
    T = TUNE[rd]
    a = pag2.mixed_anova(T, g)
    cm = {gg: T[g == gg].mean(0) for gg in ("Con", "EAA")}
    peaks = {gg: GRID[np.argmax(v)] for gg, v in cm.items()}
    ok = all(p == 0.5 for p in peaks.values()) and a["A"][3] < 0.05 and a["group"][3] < 0.05 and cm["EAA"].mean() < cm["Con"].mean() and a["group:A"][3] >= 0.05
    print(f"{rd[0]} {rd[1]:4s}: peaks {peaks}; assignment F({a['A'][1]},{a['A'][2]}) = {a['A'][0]:.2f}, p = {a['A'][3]:.1e}; "
          f"group F({a['group'][1]},{a['group'][2]}) = {a['group'][0]:.2f}, p = {a['group'][3]:.3f}; group x assignment F = {a['group:A'][0]:.2f}, p = {a['group:A'][3]:.3f}"
          f" -> {'reproduced' if ok else 'not'}")
T = TUNE[ADOPTED]
print("adopted reading, curve Con", T[g == "Con"].mean(0).round(3), "EAA", T[g == "EAA"].mean(0).round(3))
plt.figure(figsize=(5, 3))
for gg, col in (("Con", "gray"), ("EAA", "navy")):
    plt.errorbar(GRID, T[g == gg].mean(0), T[g == gg].std(0, ddof=1) / np.sqrt((g == gg).sum()), color=col, label=gg)
plt.axvline(0.375, ls="--", color="k", lw=0.5); plt.xlabel("uncertainty assignment"); plt.ylabel("threat beta"); plt.legend(); plt.show()
print("plotted range %.2f to %.2f" % (T.mean(0).min() if False else min(T[g == gg].mean(0).min() for gg in ("Con", "EAA")), max(T[g == gg].mean(0).max() for gg in ("Con", "EAA"))))

**Result: reproduced, resting on an assumption.** Under the adopted reading (d, 0/0 trials set to
1) both curves peak at 0.500, the assignment effect is F(8,632) = 18.69, p = 3e-25, the group effect
F(1,79) = 4.15, p = 0.045 with EAA lower (curve 0.34 to 0.60 against 0.56 to 1.00), and there is no
interaction (p = 0.22). The peaks are at 0.500 under ten of the twelve readings and the assignment
effect holds under all, but the group effect is significant only with the script's firing (d) and
0/0 trials dropped or set to 1 (p = 0.046 and 0.045); under the claims file's readings (a to c) it
is p = 0.07 to 0.24, and under those the claim is not reproduced. The outcome therefore rests on the
adopted combination, which is the script's (lines 2162 to 2171, printed in the data
check: `regress` of `fire.z.s1.interval` on `cer.cue`, which stores 1 for 0/0 trials) and the one closest to the paper's statistics (its
assignment and group F are 18.69 and 4.15 against the paper's 18.88 and 4.01).

## figs4-peak-assignment-same-pattern

> Using the peak uncertainty assignment (0.500) in place of the actual probability gave the same
> pattern of threat probability signaling in cue-excited neurons and of fear output signaling in
> cue-inhibited neurons.

In [ ]:
B05 = all_betas(0.5)
def s4(B):
    out = {}
    for rd, (FO, TP) in B.items():
        me, mi = lab == "exc", lab == "inh"
        tp = TP[me].mean(1); fo = FO[mi][:, 5:].mean(1)
        out[rd] = (stats.ttest_ind(tp[grp[me] == "Con"], tp[grp[me] == "EAA"]), stats.ttest_ind(fo[grp[mi] == "Con"], fo[grp[mi] == "EAA"]))
    return out
a375, a5 = s4(BETAS), s4(B05)
for rd in READINGS:
    same = all((x.pvalue < 0.05) == (y.pvalue < 0.05) and np.sign(x.statistic) == np.sign(y.statistic) for x, y in zip(a375[rd], a5[rd]))
    print(f"{rd[0]} {rd[1]:4s}: exc whole-cue TP  0.375 t = {a375[rd][0].statistic:.2f} p = {a375[rd][0].pvalue:.3f} | 0.5 t = {a5[rd][0].statistic:.2f} p = {a5[rd][0].pvalue:.3f};"
          f"  inh last-5 s FO  0.375 t = {a375[rd][1].statistic:.2f} p = {a375[rd][1].pvalue:.3f} | 0.5 t = {a5[rd][1].statistic:.2f} p = {a5[rd][1].pvalue:.3f}  same: {same}")

**Result: reproduced.** Under the adopted reading the cue-excited threat probability difference
is significant with both values (0.375 p = 0.037, 0.5 p = 0.040, Control higher), and so is the
cue-inhibited fear output difference (p = 0.021 and 0.011, EAA more negative). The significance of
one of the two tests changes between 0.375 and 0.5 under three other readings (b drop, b one, d
zero), and is the same under the other nine. Readings (a) and (c) of the claims file give the same
significance and direction with both values, so the outcome does not rest on the assumption. Under
(a) to (c) the cue-excited threat probability difference is not significant with either value, so
for that test the two values agree in giving no difference.

## fig5efg-inhibited-fear-output

> Control cue-inhibited neurons signaled a mix of threat probability and fear output (negative
> betas for both). In EAA cue-inhibited neurons the signals diverged in the second half of the cue,
> fear output persisting and threat probability diminishing (group by regressor by interval
> interaction); fear output betas in the last 5 s were more negative in EAA, threat probability
> betas did not differ.

In [ ]:
def fig5efg(B, L=None):
    L = lab if L is None else L
    m = L == "inh"; g = grp[m]; rows = {}
    for rd, (FO, TP) in B.items():
        a = pag2.mixed_anova(np.stack([FO[m], TP[m]], 1), g)
        fo, tp = FO[m][:, 5:].mean(1), TP[m][:, 5:].mean(1)
        t1 = stats.ttest_ind(fo[g == "Con"], fo[g == "EAA"]); t2 = stats.ttest_ind(tp[g == "Con"], tp[g == "EAA"])
        f = a["group:AB"]
        full = f[3] < 0.05 and t1.pvalue < 0.05 and fo[g == "EAA"].mean() < fo[g == "Con"].mean() and t2.pvalue >= 0.05
        part = (not full) and t1.pvalue < 0.05 and fo[g == "EAA"].mean() < fo[g == "Con"].mean()
        rows[rd] = "reproduced" if full else "partial" if part else "not"
        print(f"{rd[0]} {rd[1]:4s}: group x regressor x interval F({f[1]},{f[2]}) = {f[0]:.2f}, p = {f[3]:.3f}; last 5 s FO Con {fo[g=='Con'].mean():.3f} "
              f"EAA {fo[g=='EAA'].mean():.3f} t = {t1.statistic:.2f} p = {t1.pvalue:.3f}; TP t = {t2.statistic:.2f} p = {t2.pvalue:.3f} -> {rows[rd]}")
    return rows
r5e = fig5efg(BETAS)
FO, TP = BETAS[ADOPTED]; m = lab == "inh"
for g in ("Con", "EAA"):
    print(g, "adopted reading, mean betas per interval: TP", TP[m & (grp == g)].mean(0).round(2), "FO", FO[m & (grp == g)].mean(0).round(2))
print("\nvariant, exact-rate selection"); fig5efg(all_betas(L=lab_exact), L=lab_exact)

**Result: reproduced.** Under the adopted reading the three-way interaction is F(9,1170) = 2.15, p
= 0.023, the last-5 s fear output beta is more negative in EAA (-0.386 against -0.220, t = 2.34, p
= 0.021) and the threat probability betas do not differ (p = 0.41). Readings (a) and (c) with every
treatment of 0/0 trials also meet all three parts, so the outcome does not rest on the assumption;
reading (b) misses the fear output part (p = 0.054 to 0.13). With the exact-rate selection the
three-way interaction is not significant under any reading (p = 0.060 to 0.19).

## Firing and signaling (Figs. 6, 7, S5, S6)

Per neuron, the normalized firing to each cue in a window (first 1 s for cue-excited neurons, last
5 s for cue-inhibited) against a beta (whole-cue mean for Fig. 6 and S5, last-5 s mean for Fig. 7
and S6), Pearson correlation per group and cue; groups compared by Fisher r-to-z, one- and
two-tailed. Each reading of the regression is run. As a variant, the firing is also taken with the
script's normalization (mean of the cube's trial-wise Z scores), with the adopted regression
reading.

In [ ]:
def corr_table(pop, win, which, span, F_all=None, readings=None, B=None, show=True):
    F_all = NF if F_all is None else F_all
    B = BETAS if B is None else B
    res = {}
    for rd in (READINGS if readings is None else readings):
        FO, TP = B[rd]
        Bm = (TP if which == "TP" else FO)[:, span].mean(1)
        rows = {}
        for k, ty in enumerate(pag2.TYPES):
            F = F_all[:, k, pag2.WIN[win]].mean(1)
            rr = {}
            for gg in ("Con", "EAA"):
                mm = (lab == pop) & (grp == gg)
                r, p = stats.pearsonr(F[mm], Bm[mm]); rr[gg] = (r, p, int(mm.sum()))
            z, p1, p2 = pag2.fisher_r_to_z(rr["Con"][0], rr["Con"][2], rr["EAA"][0], rr["EAA"][2])
            rows[ty] = dict(Con=rr["Con"], EAA=rr["EAA"], z=(z, p1, p2))
        res[rd] = rows
    for rd in res if show else ():
        print(f"{rd[0]} {rd[1]:4s}: " + " | ".join(
            f"{ty[:3]} Con r={v['Con'][0]:.2f} p={v['Con'][1]:.2g}, EAA r={v['EAA'][0]:.2f} p={v['EAA'][1]:.2g}, Z={v['z'][0]:.2f} p1={v['z'][1]:.2g} p2={v['z'][2]:.2g}"
            for ty, v in res[rd].items()))
    return res
FULL, LAST = slice(0, 10), slice(5, 10)

### fig6-excited-firing-threat-decoupled

> In Control cue-excited neurons, danger and uncertainty firing in the first 1 s of the cue
> predicted the whole-cue threat probability beta; in EAA cue-excited neurons they did not, and the
> relationships differed between groups. Safety firing was unrelated in both groups, and that
> relationship did not differ between groups.

In [ ]:
c6 = corr_table("exc", "first1", "TP", FULL)
def judge6(v):
    dang = v["danger"]["Con"][0] > 0 and v["danger"]["Con"][1] < 0.05 and v["danger"]["EAA"][1] >= 0.05 and v["danger"]["z"][1] < 0.05
    unc = v["uncertainty"]["Con"][0] > 0 and v["uncertainty"]["Con"][1] < 0.05 and v["uncertainty"]["EAA"][1] >= 0.05 and v["uncertainty"]["z"][1] < 0.05
    saf = v["safety"]["Con"][1] >= 0.05 and v["safety"]["EAA"][1] >= 0.05 and v["safety"]["z"][1] >= 0.05
    return dang, unc, saf
for rd in READINGS:
    d_, u_, s_ = judge6(c6[rd])
    print(rd, "danger part", d_, "uncertainty part", u_, "safety part", s_)
v = c6[ADOPTED]
print("adopted reading: Control R2 danger %.3f (paper 0.56), uncertainty %.3f (paper 0.22)" % (v["danger"]["Con"][0] ** 2, v["uncertainty"]["Con"][0] ** 2))
print("\nvariant, the script's normalization of the firing, every regression reading")
c6c_all = corr_table("exc", "first1", "TP", FULL, F_all=NFc)
for rd in READINGS:
    v = c6c_all[rd]
    print(rd, "parts (danger, uncertainty, safety):", judge6(v), "; R2 Con danger %.3f, EAA %.3f; Con uncertainty %.3f, EAA %.3f"
          % (v["danger"]["Con"][0] ** 2, v["danger"]["EAA"][0] ** 2, v["uncertainty"]["Con"][0] ** 2, v["uncertainty"]["EAA"][0] ** 2))
c6c = c6c_all[ADOPTED]

**Result: partially reproduced.** With the Methods' normalization of the firing, which the claims
file fixes, the danger part holds under every reading: under the adopted one, Control r = 0.51 (R2
= 0.26, p = 0.003), EAA r = -0.00 (p = 0.98), Z = 2.37, one-tailed p = 0.009. The uncertainty part
fails under every reading, as the Control uncertainty correlation is not significant (adopted r =
0.09, R2 = 0.008, p = 0.62), and the safety part fails under every reading, as EAA safety firing
correlates negatively with the beta (adopted p = 0.040). The Control R2 values (0.26 and 0.008)
are far below the paper's 0.56 and 0.22. With the script's normalization of the firing, run as a
full variant, every part holds under nine of the twelve regression readings (all but the three
with firing b), and under the adopted one the statistics are the paper's: Control danger R2 =
0.566, p = 7e-7 (paper 0.56, 7.5e-7), Z = 3.83 (paper 3.80); Control uncertainty R2 = 0.218, p =
0.007 (paper 0.22, 0.007), Z = 3.07 (paper 3.05). The outcome follows the Methods, as the claims
file does; under the script's normalization it would be reproduced.

### figs5-excited-firing-fear-output-unrelated

> In cue-excited neurons, cue firing did not predict fear output signaling for any cue in either
> group, and the relationships did not differ between groups.

In [ ]:
def any_sig5(v):
    return any(v[ty][gg][1] < 0.05 for ty in v for gg in ("Con", "EAA")) or any(v[ty]["z"][1] < 0.05 for ty in v)
cs5 = corr_table("exc", "first1", "FO", FULL)
for rd in READINGS:
    v = cs5[rd]
    print(rd, "any correlation or one-tailed comparison significant:", any_sig5(v),
          "; smallest correlation p %.3f, smallest comparison p (one-tailed) %.3f" % (min(v[ty][gg][1] for ty in v for gg in ("Con", "EAA")), min(v[ty]["z"][1] for ty in v)))
print("\nvariant, the script's normalization, every regression reading")
cs5c = corr_table("exc", "first1", "FO", FULL, F_all=NFc)
for rd in READINGS:
    v = cs5c[rd]
    print(rd, "any significant:", any_sig5(v), "; smallest correlation p %.3f, smallest comparison p (one-tailed) %.3f"
          % (min(v[ty][gg][1] for ty in v for gg in ("Con", "EAA")), min(v[ty]["z"][1] for ty in v)))

**Result: reproduced.** With the Methods' normalization, under every reading none of the six
correlations and none of the three comparisons is significant (under the adopted one the smallest
correlation p is 0.23 and the smallest one-tailed comparison p 0.13). With the script's
normalization the adopted reading also has none (smallest comparison p 0.059), but six of the
twelve readings have one comparison with one-tailed p < 0.05 (smallest 0.010).

### fig7-inhibited-firing-fear-output

> In cue-inhibited neurons, last-5 s danger firing did not predict the last-5 s fear output beta in
> either group; uncertainty firing predicted it positively in EAA but not in Control neurons, and
> safety firing predicted it negatively in both groups; none of these relationships differed
> between groups.

In [ ]:
def judge7(v):
    return dict(
        danger=v["danger"]["Con"][1] >= 0.05 and v["danger"]["EAA"][1] >= 0.05,
        uncertainty=v["uncertainty"]["EAA"][0] > 0 and v["uncertainty"]["EAA"][1] < 0.05 and v["uncertainty"]["Con"][1] >= 0.05,
        safety=all(v["safety"][gg][0] < 0 and v["safety"][gg][1] < 0.05 for gg in ("Con", "EAA")),
        comparisons=v["danger"]["z"][1] >= 0.05 and v["uncertainty"]["z"][1] >= 0.05 and v["safety"]["z"][2] >= 0.05)
c7 = corr_table("inh", "last5", "FO", LAST)
for rd in READINGS:
    print(rd, judge7(c7[rd]))
print("\nvariant, the script's normalization, every regression reading")
c7c = corr_table("inh", "last5", "FO", LAST, F_all=NFc)
for rd in READINGS:
    print(rd, judge7(c7c[rd]))
print("\nvariant, other firing windows, adopted regression reading. The legend and Results give the last 5 s; the script computes"
      " cue-inhibited firing only over the first 5 s and the whole cue (vlpagAnalysis.m, 'Find mean firing in first 5 seconds' and 'for whole cue').")
print("   paper: danger Con R2 0.004 p 0.67, EAA 0.02 p 0.24, Z 1.04 | uncertainty Con 0.04 p 0.16, EAA 0.20 p 2.6e-05, Z 1.43 | safety Con 0.14 p 0.009, EAA 0.22 p 8.9e-06, Z 0.60")
for win in ("last5", "first5", "cue"):
    for nm, Fa in (("Methods", NF), ("script", NFc)):
        v = corr_table("inh", win, "FO", LAST, F_all=Fa, readings=[ADOPTED])[ADOPTED]
        print(f"   firing {win:6s} {nm:7s} normalization: " + " | ".join(
            f"{ty[:3]} Con R2 {v[ty]['Con'][0] ** 2:.3f} p {v[ty]['Con'][1]:.2g}, EAA R2 {v[ty]['EAA'][0] ** 2:.3f} p {v[ty]['EAA'][1]:.2g}, Z {v[ty]['z'][0]:.2f}"
            for ty in v) + "; parts " + str({k: bool(x) for k, x in judge7(v).items()}))
print("\nvariant, firing over the first 5 s, every regression reading: parts that hold")
for nm, Fa in (("Methods", NF), ("script", NFc)):
    c7f = corr_table("inh", "first5", "FO", LAST, F_all=Fa, show=False)
    for rd in READINGS:
        v = c7f[rd]
        print(f"   {nm:7s} normalization, {rd[0]} {rd[1]:4s}: " + str({k: bool(x) for k, x in judge7(v).items()})
              + f"; Control uncertainty r = {v['uncertainty']['Con'][0]:.2f}, p = {v['uncertainty']['Con'][1]:.2g}")


**Result: not reproduced.** With the last-5 s firing the claims file names, the Control uncertainty
correlation is positive and significant under every reading and under both normalizations (Methods'
normalization r = 0.35 to 0.73, p <= 0.015, the weakest under firing c; adopted, Methods r = 0.71, R2 = 0.51, p = 1e-8; script R2 = 0.49, p = 3e-8; the paper R2 = 0.04, p = 0.16), so the
uncertainty part fails, and with it the condition for a partial outcome. Under the adopted reading
the danger correlations are not significant (p = 0.054 and 0.15), both safety correlations are
negative and significant (p = 0.026 and 0.004), and the uncertainty comparison is significant
one-tailed (Z = 2.06, p = 0.020), against the claim.

With the firing over the first 5 s of the cue, the window the script computes for cue-inhibited
neurons (data check), every part holds under the adopted reading with both normalizations, and with the script's normalization the
statistics are the paper's: Control uncertainty R2 = 0.042, p = 0.16 (paper 0.04, 0.16), EAA R2 =
0.193, p = 2.9e-5 (paper 0.20, 2.6e-5), Z = -1.41 (paper 1.43); safety Control R2 = 0.14, p =
0.009 and EAA R2 = 0.218, p = 7.7e-6 (paper 0.14, 0.009 and 0.22, 8.9e-6), Z = 0.60 (paper 0.60);
danger Control R2 = 0.003, p = 0.70 and EAA R2 = 0.014, p = 0.28 (paper 0.004, 0.67 and 0.02, 0.24).
Under all twelve regression readings, first-5 s firing meets every part with
firing (b) or (d) (all six with the script's normalization; five of six with the Methods', b with
0/0 trials set to 1 failing the safety part); with (a) the Control uncertainty correlation stays
significant (p <= 0.004), and with (c) the danger and safety parts fail. The paper's Fig. 7 was evidently computed on first-5 s firing, while its legend and Results say the
last 5 s. The claims file follows the legend, so the outcome stays not reproduced; the mismatch is
recorded as a finding.

### figs6-inhibited-firing-threat-probability

> In cue-inhibited neurons, cue firing also predicted threat probability signaling in both groups,
> with no difference between Control and EAA for any relationship.

In [ ]:
def judge_s6(v):
    return all(any(v[ty][gg][1] < 0.05 for ty in v) for gg in ("Con", "EAA")), all(v[ty]["z"][1] >= 0.05 for ty in v), all(v[ty]["z"][2] >= 0.05 for ty in v)
cs6 = corr_table("inh", "last5", "TP", LAST)
for rd in READINGS:
    v = cs6[rd]
    print(rd, "a cue with p < 0.05 in each group, no comparison one-tailed, none two-tailed:", judge_s6(v),
          "; comparisons one-tailed p", [round(v[ty]["z"][1], 3) for ty in v], "two-tailed", [round(v[ty]["z"][2], 3) for ty in v])
print("\nvariant, the script's normalization, every regression reading")
cs6c = corr_table("inh", "last5", "TP", LAST, F_all=NFc)
for rd in READINGS:
    v = cs6c[rd]
    print(rd, judge_s6(v), "; comparisons one-tailed p", [round(v[ty]["z"][1], 3) for ty in v])

**Result: reproduced.** Under the adopted reading danger firing predicts the threat probability
beta in both groups (r = 0.73 and 0.84) and no comparison is significant (smallest p 0.052
one-tailed, 0.10 two-tailed). Under the script's firing with 0/0 trials set to 0 or dropped, the
danger comparison is significant one-tailed (p = 0.036 and 0.038) but not two-tailed; under
readings (a) to (c) no comparison is significant. With the script's normalization every
part holds under all twelve readings (smallest one-tailed comparison p 0.096).

### The design that is not of full rank

One cue-inhibited neuron (section Neurons in the regressions) comes from a session in which
the rat never poked during a cue, so with 0/0 trials set to 1 or dropped its suppression ratio is 1
on every trial and equals the constant. `numpy.linalg.lstsq` then returns the minimum-norm betas,
which split the constant between the two columns; Matlab `regress`, which the authors' script uses,
sets the fear output beta to 0. Below, the adopted reading is re-run with Matlab's solution
(`pag2.regress_betas(..., basic=True)`) for the claims that use cue-inhibited neurons.


In [ ]:
rd_idx = [j for j in np.where(resp)[0] if not full_rank(U[j], "one")]
BM = {ADOPTED: tuple(a.copy() for a in BETAS[ADOPTED])}
for j in rd_idx:
    fo, tp = pag2.interval_betas(U[j], ADOPTED[0], ADOPTED[1], cube_z=ZS1[:, :, j], basic=True)
    print(U[j]["name"], grp[j], lab[j], "fear output betas, minimum norm:", BETAS[ADOPTED][0][j].round(2), "Matlab:", fo.round(2))
    BM[ADOPTED][0][j], BM[ADOPTED][1][j] = fo, tp
print("\nfig5efg-inhibited-fear-output, Matlab solution:"); fig5efg(BM)
BM05 = {ADOPTED: tuple(a.copy() for a in B05[ADOPTED])}
for j in rd_idx:
    BM05[ADOPTED][0][j], BM05[ADOPTED][1][j] = pag2.interval_betas(U[j], ADOPTED[0], ADOPTED[1], 0.5, ZS1[:, :, j], basic=True)
x375, x5 = s4(BM)[ADOPTED][1], s4(BM05)[ADOPTED][1]
print(f"figs4-peak-assignment-same-pattern, inh last-5 s FO, Matlab solution: 0.375 t = {x375.statistic:.2f} p = {x375.pvalue:.3f} | 0.5 t = {x5.statistic:.2f} p = {x5.pvalue:.3f}")
print("\nfig7-inhibited-firing-fear-output, Matlab solution:")
v = corr_table("inh", "last5", "FO", LAST, readings=[ADOPTED], B=BM)[ADOPTED]
print("   parts", {k: bool(x) for k, x in judge7(v).items()})
print("   script's normalization:", {k: bool(x) for k, x in judge7(corr_table("inh", "last5", "FO", LAST, F_all=NFc, readings=[ADOPTED], B=BM, show=False)[ADOPTED]).items()})
print("\nfigs6-inhibited-firing-threat-probability, Matlab solution:")
v = corr_table("inh", "last5", "TP", LAST, readings=[ADOPTED], B=BM)[ADOPTED]
print("   ", judge_s6(v), "; comparisons one-tailed p", [round(v[ty]["z"][1], 3) for ty in v])


With Matlab's solution the neuron's fear output betas are 0 in every interval. Fig. 5E to G
gives F(9,1170) = 2.14, p = 0.024, last-5 s fear output t = 2.30, p = 0.023 and threat p = 0.41
(against 2.15, 0.021 and 0.41); the Fig. S4 cue-inhibited test has p = 0.023 and 0.012 with 0.375
and 0.5; the parts of Fig. 7 that hold are the same under both normalizations, and Fig. S6 holds
(smallest one-tailed comparison p = 0.052). No outcome changes.


## Summary

| Claim | Outcome |
|---|---|
| fig1bc-adversity-slows-weight-gain | blocked, data missing |
| fig1ef-acquisition-equivalent-final | blocked, data missing |
| fig2b-baseline-poke-equivalent | reproduced |
| fig2c-recording-discrimination | reproduced |
| fig2d-cue-responsive-counts | reproduced |
| fig2d-higher-proportion-responsive | reproduced |
| fig2ef-baseline-firing | reproduced |
| fig4ab-excited-temporal-pattern | partially reproduced (bregma part not testable) |
| fig4c-excited-inflated-onset | reproduced |
| fig4de-inhibited-temporal-pattern | partially reproduced (bregma part not testable) |
| fig4f-inhibited-danger-decrease | reproduced |
| fig4f-inhibited-danger-uncertainty-discrimination | reproduced |
| fig5abc-excited-reduced-threat-probability | partially reproduced |
| fig5d-threat-tuning-curve | reproduced (assumption) |
| figs4-peak-assignment-same-pattern | reproduced |
| fig5efg-inhibited-fear-output | reproduced |
| fig6-excited-firing-threat-decoupled | partially reproduced |
| figs5-excited-firing-fear-output-unrelated | reproduced |
| fig7-inhibited-firing-fear-output | not reproduced |
| figs6-inhibited-firing-threat-probability | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's: the selection uses the cube's stored rates, truncated to
whole Hz, as the authors' script does (with exact rates the Control inhibited count is 54, not 48);
the regressions are run under twelve readings of the trial-wise firing and of trials without
pokes, and every regression claim is judged under one of them, the script's own (d, 0/0 trials set
to 1), which also gives the paper's statistics most closely; the outcome of Fig. 5D rests on it, as the
claims file's readings give a different outcome, while those of Figs. 5A to C and S4 are the same
under the claims file's readings; the ANOVAs with
bregma level could not be run; the ANOVAs and t tests use all selected neurons, so several error
degrees of freedom are larger than the paper's (for example 79 against 65 for the cue-excited
regression ANOVA). The population analyses follow the Methods' normalization, as the claims file
fixes it; the authors' script instead averages the cube's trial-wise Z scores, and with that the
statistics of Figs. 4C, 4F and 6 are the paper's to the second digit, and Fig. 6 would be
reproduced. The paper's Fig. 7 statistics follow from firing over the first 5 s of the cue with the
script's normalization and the adopted regression reading, where its legend says the last 5 s; with the last 5 s, as the claims file
has it, Fig. 7 is not reproduced. The paper's smaller degrees of freedom for several tests are not
explained by trials without pokes, missing betas or any step in the script. One cue-inhibited
neuron has a design not of full rank; the regressions give it minimum-norm betas where Matlab
`regress` gives a fear output beta of 0, and with Matlab's solution no outcome changes.

What was found about the data set: the units come from 182 sessions (74 Control, 108 EAA), not the
stated 179; group membership and the event codes are documented only in the script (the codes
change meaning with the behaviour program); the cube stores firing rates truncated to integers;
`poke.base` is the 20 s baseline in pokes per minute, not the 2 s the description states; trials
without pokes are stored with a suppression ratio of 1; the description gives the uncertainty
shock probability as 0.25; there is no bregma level per rat, no body weight and no pre-surgery
session.